# PIE's benchmark

Trains PWPA on PIE's benchmark of unseen knockdowns in seen contexts (Replogle–Nadig: K562, RPE1, HepG2, Jurkat) and scores it with PIE's metric code, next to the values in PIE's paper.

- Data and splits: PIE's processed arrays and its four published folds. Each fold is named after the cell line it holds out, and its other three lines are scored.
- Priors: PIE's five genetic knowledge sources, the same gene information PIE gets.
- One PWPA per fold and cell line, trained on that line's training knockdowns, with RFE choosing the priors and early stopping on PIE's validation knockdowns.
- Scores: the mean over cell lines, then the mean and standard error over folds.

PIE's metric code comes from the `pie` package ([ArcInstitute/pie](https://github.com/ArcInstitute/pie) at `21a526a`), so the kernel needs PIE installed or PIE's `src/` on `PYTHONPATH`.

Writes `results/pie_benchmark_scores.tsv` and the main figure, `figures/main_figure.png`.

In [ ]:
import json
import os
import sys

import numpy as np
import pandas as pd
import torch

sys.path.append("../utils")
from paths import data_dir
from pie_scoring import METRICS, load, pie_metrics, responses

sys.path.append("../1.model")
import pwpa

torch.set_num_threads(int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count())))

In [ ]:
# PIE's Table 4 as published (mean and sd over folds; se = sd / 2); the source column says where PWPA's rows come from
scores = pd.read_csv("results/pie_benchmark_scores.tsv", sep="\t")
scores.pivot(index="model", columns="metric", values="mean")

In [ ]:
pie_dir = data_dir("pie_benchmark")
d = load(pie_dir / "preprocessed")
priors = pwpa.load_priors(data_dir("priors"))  # PIE's five genetic knowledge sources

units = []
for fold in ["hepg2", "jurkat", "k562", "rpe1"]:
    split = {part: json.loads((pie_dir / "splits" / "unseen_pert" / fold / f"{part}.json").read_text()) for part in ("train", "val", "test")}
    for key in split["test"]:  # "replogle.<line>"
        line = key.split(".", 1)[1]
        train, val, test = (np.flatnonzero((d.context == line) & np.isin(d.pert, split[part][key])) for part in ("train", "val", "test"))
        fit = np.concatenate([train, val])  # validation knockdowns stop the fit early but are never partners
        best = pwpa.choose(pwpa.rfe(priors, [(d.pert[fit], responses(d, fit))], held_out=d.pert[val]))
        chosen = [priors[name] for name in best["priors"]]
        delta, W = pwpa.predict(chosen, best["scale"], best["gates"], d.pert[test], d.pert[train], responses(d, train))
        p_de, lfc = pwpa.predict_de(W, d.de[train], d.lfc[train], own=d.own[train], tested=d.tested[train].any(0))
        units.append(pie_metrics(d, test, p_de, lfc, delta).rename(fold))
        print(fold, line, best["priors"], flush=True)

per_fold = pd.DataFrame(units).groupby(level=0).mean()  # a fold's value: the mean over its three lines, NaN skipped
pwpa_rows = pd.DataFrame({
    "model": "pwpa", "metric": METRICS, "mean": per_fold.mean().values,
    "se": (per_fold.std(ddof=1) / np.sqrt(len(per_fold))).values, "source": "0.pie_benchmark.ipynb",
})
scores = pd.concat([scores.query("model != 'pwpa'"), pwpa_rows], ignore_index=True)
scores.to_csv("results/pie_benchmark_scores.tsv", sep="\t", index=False)
scores.pivot(index="model", columns="metric", values="mean")

In [ ]:
import matplotlib.image as mpimg
import matplotlib.pyplot as plt

# the main figure: (A) how PWPA is trained, from 1.model; (B) PWPA against PIE's published results
METRICS = {
    "binary_auprc": "AUPRC",
    "sig_jaccard": "Jaccard",
    "direction_match": "DE direction match",
    "spearman_nsig": "Spearman effect size",
    "spearman_lfc": "Spearman LFC",
    "discrimination_score_l1": "Discrimination L1",
}
MODELS = {
    "context_mean": ("Context\nmean", "#8d97a8"),
    "compass_n": ("COMPASS-N", "#c9822f"),
    "presage": ("PRESAGE", "#7d9a4a"),
    "tabicl": ("TabICL", "#3f9c93"),
    "pie": ("PIE", "#2c6aa8"),
    "pwpa": ("PWPA", "#a03a8c"),
}
plt.rcParams.update({"font.size": 9, "axes.titlesize": 11, "axes.titleweight": "bold"})

fig = plt.figure(figsize=(12, 15.6), layout="constrained")
top, bottom = fig.subfigures(2, 1, height_ratios=[7.0, 8.6])
diagram = top.subplots()
diagram.imshow(mpimg.imread("../1.model/figures/rfe_training.png"))
diagram.axis("off")
top.text(0.0, 1.0, "A", fontsize=20, fontweight="bold", va="top")

bottom.suptitle("PWPA and the models in PIE's paper: unseen knockdowns in seen contexts (Replogle–Nadig)", fontweight="bold")
bottom.text(0.0, 1.0, "B", fontsize=20, fontweight="bold", va="top")
for ax, (metric, title) in zip(bottom.subplots(2, 3).flat, METRICS.items()):
    cell = scores.query("metric == @metric").set_index("model")
    for x, (model, (_, color)) in enumerate(MODELS.items()):
        mean, se = cell.loc[model, ["mean", "se"]]
        if pd.isna(mean):  # the context mean has no Spearman effect size
            ax.text(x, 0.01, "NaN", ha="center", va="bottom", fontsize=8, color="#5a6478")
            continue
        ax.bar(x, mean, yerr=se, color=color, width=0.72, zorder=2, error_kw={"lw": 1, "capsize": 3})
        ax.text(x, mean + se, f"{mean:.3f}", ha="center", va="bottom", fontsize=7.5, family="monospace")
    ax.set_title(title)
    ax.set_xticks(range(len(MODELS)), [name.replace("\n", " ") for name, _ in MODELS.values()], rotation=35, ha="right", fontsize=8.5)
    ax.get_xticklabels()[-1].set_fontweight("bold")
    ax.grid(axis="y", color="#e3e6ec", zorder=0)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="y", length=0)
    ax.set_xlim(-0.6, len(MODELS) - 0.4)
fig.savefig("figures/main_figure.png", dpi=200, bbox_inches="tight", facecolor="white")